# Test Words Model

In [ ]:
import os, cv2, numpy as np, mediapipe as mp, time
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from tensorflow.keras.models import load_model

BASE           = r'Sign_to_Sentence Project'
gesture_model  = load_model(os.path.join(BASE,'gesture_word_model.h5'), compile=False)
GESTURE_LABELS = np.load(os.path.join(BASE,'gesture_labels.npy'), allow_pickle=True).tolist()
print('Classes:', GESTURE_LABELS)

COLORS = {
    'hi'              : (0,  215,255),
    'good'            : (0,  200,100),
    'thank you'       : (255,150,  0),
    'how are you'     : (100,200,255),
    'computer vision' : (180,  0,255),
}
THRESHOLD   = 0.85
STABLE_NEED = 4
HOLD_SECS   = 0.6

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
hands    = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.6, min_tracking_confidence=0.6)

def lm_flat(hand_lm):
    pts = []
    for lm in hand_lm.landmark: pts.extend([lm.x, lm.y, lm.z])
    return np.array(pts, dtype=np.float32).reshape(1,-1)

def stable(buf, val, n):
    buf.append(val)
    if len(buf) > n: buf.pop(0)
    return buf[0] if len(buf)==n and len(set(str(x) for x in buf))==1 else None

def txt(frame, s, x, y, sc=0.5, col=(200,200,200), th=1):
    cv2.putText(frame, str(s), (int(x),int(y)), cv2.FONT_HERSHEY_SIMPLEX, sc, col, th)

# open camera
cap = None
for idx in [0,1]:
    for backend in [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]:
        try:
            c = cv2.VideoCapture(idx, backend)
            if c.isOpened():
                ret, f = c.read()
                if ret and f is not None and f.size > 0:
                    cap = c
                    print(f'Camera: idx={idx} backend={backend}')
                    break
                c.release()
        except: pass
    if cap: break

if cap is None: raise RuntimeError('No camera found!')
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
print('Running — Q=quit  C=clear')

buf, confirmed = [], []
gs = {'current':None,'hold_start':None,'confirmed':None,'progress':0.0}

while True:
    ret, frame = cap.read()
    if not ret or frame is None: time.sleep(0.05); continue

    frame  = cv2.flip(frame, 1)
    result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    h, w   = frame.shape[:2]

    ov = frame.copy()
    cv2.rectangle(ov,(0,0),(w-320,75),(20,20,28),-1)
    cv2.rectangle(ov,(w-320,0),(w,h),(15,15,20),-1)
    cv2.addWeighted(ov,0.88,frame,0.12,0,frame)

    txt(frame,'WORDS MODEL TEST',16,45,1.0,(255,255,255),2)
    txt(frame,'Q=quit  C=clear', 16,66,0.42,(100,100,100),1)

    stb = None
    if result.multi_hand_landmarks:
        hand_lm = result.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
        preds  = gesture_model(lm_flat(hand_lm), training=False).numpy()[0]
        g_idx  = int(np.argmax(preds))
        g_conf = float(preds[g_idx])
        raw    = GESTURE_LABELS[g_idx] if g_conf >= THRESHOLD else None
        stb    = stable(buf, raw, STABLE_NEED)

        if stb:
            gcol = COLORS.get(stb,(200,200,200))
            txt(frame, stb.upper(), 30, 170, 1.6, gcol, 4)
            txt(frame, str(int(g_conf*100))+'%', 30, 210, 0.9, (150,150,150), 2)
            bw = int((w-340)*g_conf)
            cv2.rectangle(frame,(20,h-44),(w-330,h-30),(35,35,45),-1)
            cv2.rectangle(frame,(20,h-44),(20+bw,h-30),gcol,-1)
            if gs['current'] != stb:
                gs.update({'current':stb,'hold_start':time.time(),'confirmed':None,'progress':0.0})
            held = time.time() - (gs['hold_start'] or time.time())
            gs['progress'] = min(held/HOLD_SECS, 1.0)
            hw = int((w-340)*gs['progress'])
            cv2.rectangle(frame,(20,h-26),(w-330,h-12),(35,35,45),-1)
            cv2.rectangle(frame,(20,h-26),(20+hw,h-12),gcol,-1)
            txt(frame,'Hold: '+str(int(gs['progress']*100))+'%',22,h-52,0.38,(120,120,120))
            if held >= HOLD_SECS and gs['confirmed'] != stb:
                confirmed.append(stb)
                gs.update({'confirmed':stb,'hold_start':None,'progress':0.0})
                print('[OK]', stb)
        else:
            gs.update({'current':None,'hold_start':None,'confirmed':None,'progress':0.0})
            txt(frame,'Not recognized',30,160,0.7,(60,60,90),2)
    else:
        gs.update({'current':None,'hold_start':None,'confirmed':None,'progress':0.0})
        buf.clear()
        txt(frame,'Show hand to camera',30,160,0.9,(60,60,100),2)

    txt(frame,'CONFIRMED:',w-305,38,0.48,(130,130,130))
    for j,word in enumerate(confirmed[-9:]):
        txt(frame,'- '+word,w-305,68+j*30,0.5,COLORS.get(word,(200,200,200)),2)

    cv2.line(frame,(w-320,h-185),(w,h-185),(45,45,55),1)
    txt(frame,'AVAILABLE:',w-305,h-168,0.40,(100,100,120))
    cur = gs['current']
    for j,g in enumerate(GESTURE_LABELS):
        col    = COLORS.get(g,(150,150,150))
        active = (g==cur)
        if active: cv2.rectangle(frame,(w-315,h-158+j*28),(w-5,h-134+j*28),(30,30,42),-1)
        cv2.circle(frame,(w-308,h-148+j*28),5,col,-1 if active else 1)
        txt(frame,g,w-298,h-143+j*28,0.40,col if active else (75,75,85),2 if active else 1)

    cv2.imshow('Words Model Test',frame)
    key = cv2.waitKey(1)&0xFF
    if key==ord('q'): break
    elif key==ord('c'):
        confirmed.clear(); buf.clear()
        gs.update({'current':None,'hold_start':None,'confirmed':None,'progress':0.0})

cap.release()
cv2.destroyAllWindows()
print('Done! Confirmed:', confirmed)
